# Task3 : Build a Chatbot using Hugging Face Transformers

Step-1:Install the requirements

In [ ]:
!pip install -q -U "transformers>=4.37.0" accelerate

Step-2:Importing the packages

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Step-3:Loading the model

Here we are using Qwen which is a lightweight instruction-tuned language model from Alibaba under the Qwen (Tongyi) series.

It has 0.5 billion parameter (0.5B) transformer model,designed for instruction-following tasks (chat, Q&A, basic reasoning) and it is part of the newer Qwen2.5 family

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!



Step-4: Defining response function

the response function Takes user input + history and Returns model response + updated history.

Here,the input is take, then adds it to the history, generates a response and also adds that to the history

In [ ]:
def generate_response(user_input, chat_history):
    #Add user message to the history
    chat_history.append({"role": "user", "content": user_input})
    #Formats the conversation using chat template
    input_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    #it generates the response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    #decodes only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    #Adds the bot response to history
    chat_history.append({"role": "assistant", "content": response})
    return response, chat_history

Step-5:defining loop for chatbot

this fucntion keeps the bot running until we dont type exit or quit.

In [ ]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history = []

    while True:
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day 👋")
            break

        # Generate response
        reply, chat_history = generate_response(user_input, chat_history)

        print(f"Chatbot: {reply}\n")


Chatbot Workflow:

User Input → Formatting → Tokenization → Model Generation → Response Decoding → Display Output → Repeat Until Exit

In [ ]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It encompasses a wide range of technologies and techniques used to create intelligent machines that can perform tasks that typically require human intelligence.

Key aspects of AI include:

1. Machine learning: The ability of machines to learn from data without being explicitly programmed.
2. Natural language processing (NLP): The ability of machines to understand and interpret human language.
3. Computer vision: The ability of machines to perceive and respond to visual information.
4. Robotics: The ability of machines to mimic human-like actions and behaviors.
5. Autonomous systems: Machines that operate independently and make decisions based on predefined rules or algorithms.
6. Expert systems:

You: Who created python?
Chatbot: Python 